<a href="https://colab.research.google.com/github/kab-s/flyrank-page-priority-pipeline/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kab-s/flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)


In [17]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane (or freestyle) and why

**Lane:** Page Priority Pipeline: Risk × Impact Ranking

I am building a global, client-agnostic Page Priority Pipeline that scores every published content page with `priority = P(decline in next 30 days) × expected clicks lost`, ranking all pages from most to least urgent. Content traffic in Google search decays quietly over time. Editorial teams face portfolios of thousands of articles and cannot review them all. Ranking by risk alone floods editors with low-traffic pages losing negligible clicks; ranking by traffic alone wastes hours reviewing evergreen champions that are completely stable. Multiplying predicted probability of decline by expected traffic lost pinpoints where editorial action preserves the most search value.

In [18]:
# Lane Definition & Core Formula
lane = "Page Priority Pipeline: Risk × Impact Ranking"
formula = "priority_score = P(decline in next 30 days) * expected_clicks_lost"
print(f"Selected Lane: {lane}")
print(f"Core Formula:  {formula}")
print("Objective:     Maximize search traffic preserved per editor hour invested.")

Selected Lane: Page Priority Pipeline: Risk × Impact Ranking
Core Formula:  priority_score = P(decline in next 30 days) * expected_clicks_lost
Objective:     Maximize search traffic preserved per editor hour invested.


## 2. The question: decision, action, cost of a wrong call

1. **Decision Improved:** Which pages an editorial team must audit and refresh first each week across their portfolio.
2. **Who Acts & What They Do:** SEO specialists and content editors inspect the flagged page, update outdated facts, expand keyword depth, fix decaying user intent, and strengthen internal links.
3. **Cost of a Wrong Call:**
   - **False Positive (Type I - Wasted Effort):** An editor spends 2–4 hours rewriting an asset that was never in danger of declining, diverting capacity from genuine decay.
   - **False Negative (Type II - Missed Decay):** A high-traffic asset quietly drops multiple positions in Google search, losing thousands of monthly clicks that take months of ranking recovery to regain.

In [19]:
import pandas as pd

cost_matrix = pd.DataFrame([
    {
        "Decision Outcome": "True Positive (Timely Alert)",
        "Editorial Action": "Audit & refresh decaying asset",
        "Business Impact": "Preserves high-value search clicks before traffic collapses"
    },
    {
        "Decision Outcome": "False Positive (False Alarm)",
        "Editorial Action": "Unnecessary 2-4 hour content overhaul",
        "Business Impact": "Wasted editorial hours on stable content"
    },
    {
        "Decision Outcome": "False Negative (Missed Decay)",
        "Editorial Action": "No action taken (silent rank drop)",
        "Business Impact": "Severe compounding traffic loss; costly recovery"
    },
    {
        "Decision Outcome": "True Negative (Correct Inaction)",
        "Editorial Action": "Leave healthy pages untouched",
        "Business Impact": "Zero wasted capacity on stable evergreen content"
    }
])
cost_matrix

,Decision Outcome,Editorial Action,Business Impact
0,True Positive (Timely Alert),Audit & refresh decaying asset,Preserves high-value search clicks before traf...
1,False Positive (False Alarm),Unnecessary 2-4 hour content overhaul,Wasted editorial hours on stable content
2,False Negative (Missed Decay),No action taken (silent rank drop),Severe compounding traffic loss; costly recovery
3,True Negative (Correct Inaction),Leave healthy pages untouched,Zero wasted capacity on stable evergreen content


## 3. Quick look at the data (2-3 real numbers)

Analysis of `data/raw/content_refresh_anonymized.csv` (30,000 pages across 32 clients):
1. **Extreme Traffic Concentration (80/20 Rule):** The top 10% of pages control **82.58%** of total clicks and **76.11%** of impressions, proving that prioritizing by probability alone would misallocate scarce editorial resources to low-impact pages.
2. **High Base Rate of Decline:** **54.21%** of pages (16,262 items) are actively declining in this window, showing that decay is widespread and requires automated triage.
3. **Massive Search Volume at Stake:** Declining pages represent **54,506 clicks** (36.82% of all clicks in the period), representing substantial traffic at risk of loss.
4. **Data Trap Handled:** 1,205 rows have `avg_position = 0`. In Google Search Console, rank is 1-indexed; 0 represents missing/unobserved data, not top rank.

In [20]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

# Locate starter CSV robustly across environments
def find_csv():
    candidates = [
        Path("data/raw/content_refresh_anonymized.csv"),
        Path("../../data/raw/content_refresh_anonymized.csv"),
        Path("../../../data/raw/content_refresh_anonymized.csv"),
    ]
    for c in candidates:
        if c.exists():
            return c
    return candidates[0]

csv_path = find_csv()
df_raw = pd.read_csv(csv_path)

n_rows = len(df_raw)
n_clients = df_raw["client_id"].nunique()

# Map columns
impr_col = "impressions_last_30d"
clicks_col = "clicks_last_30d"
df_raw["is_declining_label"] = (df_raw["trend_direction"] == "down").astype(int)

# 1. Traffic concentration
df_sorted_clicks = df_raw.sort_values(clicks_col, ascending=False)
top_10_n = int(np.ceil(0.10 * n_rows))
top_10_clicks_share = df_sorted_clicks.iloc[:top_10_n][clicks_col].sum() / df_raw[clicks_col].sum()

df_sorted_impr = df_raw.sort_values(impr_col, ascending=False)
top_10_impr_share = df_sorted_impr.iloc[:top_10_n][impr_col].sum() / df_raw[impr_col].sum()

# 2. Base rate of decline
decline_count = int((df_raw["is_declining_label"] == 1).sum())
base_rate = float(df_raw["is_declining_label"].mean())

# 3. Clicks at stake
total_clicks = int(df_raw[clicks_col].sum())
declining_clicks = int(df_raw.loc[df_raw["is_declining_label"] == 1, clicks_col].sum())
declining_clicks_share = declining_clicks / total_clicks

# 4. Data trap verification
zero_pos_count = int((df_raw["avg_position"] == 0).sum())

summary_df = pd.DataFrame([
    {"Metric": "Total Analyzed Pages", "Value": f"{n_rows:,}"},
    {"Metric": "Unique Clients", "Value": f"{n_clients}"},
    {"Metric": "Top 10% Page Share of Clicks", "Value": f"{top_10_clicks_share * 100:.2f}%"},
    {"Metric": "Top 10% Page Share of Impressions", "Value": f"{top_10_impr_share * 100:.2f}%"},
    {"Metric": "Base Rate of Decline", "Value": f"{base_rate * 100:.2f}% ({decline_count:,} pages)"},
    {"Metric": "Search Clicks at Risk in Declining Pages", "Value": f"{declining_clicks:,} ({declining_clicks_share * 100:.2f}% of total)"},
    {"Metric": "Unobserved Position Trap (avg_position == 0)", "Value": f"{zero_pos_count:,} rows (treated as NaN, not rank 0)"},
])

display(summary_df.style.hide(axis='index'))

Metric,Value
Total Analyzed Pages,"30,000"
Unique Clients,32
Top 10% Page Share of Clicks,82.58%
Top 10% Page Share of Impressions,76.11%
Base Rate of Decline,"54.21% (16,262 pages)"
Search Clicks at Risk in Declining Pages,"54,506 (36.82% of total)"
Unobserved Position Trap (avg_position == 0),"1,205 rows (treated as NaN, not rank 0)"


## 4. Careful words: what I can and can't claim

### What I can Claim
- **Observed historical patterns:** "Pages displaying declining momentum over the past 30 days historically continue to lose rank with X% probability."
- **Decision support & triage:** "The pipeline ranks pages to focus human editorial review where expected traffic loss is highest."
- **Directional risk indicators:** "Pages with worsening average position and decelerating impression velocity exhibit elevated risk of continued decline."
- **No Client-Specific Memorization:** Models are evaluated across held-out clients and future time periods to ensure true generalization.

### What I cannot Claim
- **No Causal Proof:** I cannot claim that updating a page will *cause* rankings to rebound without a controlled A/B test or diff-in-diff study.
- **No 'Predicting Google':** It does not predict proprietary ranking algorithm shifts; it models empirical decay in search performance metrics.
- **No Deterministic Guarantees:** Predictions represent expected risks and probabilities under historical distributions, not certainties.


In [21]:
claims_audit = pd.DataFrame([
    {"Claim Category": "Allowed", "Claim": "Observed historical decline frequency across snapshot dates", "Justification": "Empirical measurement from historical fact logs"},
    {"Claim Category": "Allowed", "Claim": "Decision-support queue prioritizing review by expected click loss", "Justification": "Operational utility for human editorial teams"},
    {"Claim Category": "Disallowed", "Claim": "Algorithm predicts Google search ranking updates", "Justification": "External proprietary system; we only model client traffic trends"},
    {"Claim Category": "Disallowed", "Claim": "Updating this page guarantees a 25% traffic recovery", "Justification": "Causal claim requiring counterfactual randomized control trial"},
])
display(claims_audit.style.hide(axis='index'))

Claim Category,Claim,Justification
Allowed,Observed historical decline frequency across snapshot dates,Empirical measurement from historical fact logs
Allowed,Decision-support queue prioritizing review by expected click loss,Operational utility for human editorial teams
Disallowed,Algorithm predicts Google search ranking updates,External proprietary system; we only model client traffic trends
Disallowed,Updating this page guarantees a 25% traffic recovery,Causal claim requiring counterfactual randomized control trial


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.